# 03 Error analysis by device type and condition

Which device types and conditions produce the misclassifications? Based on the exported
predictions of the three class random forest (no training).

1. Per device type and group: where do its packets go (background, own group, other group)?
2. The same per address (majority vote): how many addresses of each device are found?
3. Background addresses that are predicted as a group (false positives) by manufacturer.
4. Errors over time (two hour blocks in domain, 10 minute windows on the university capture).

In [ ]:
import sys, pathlib, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(pathlib.Path.cwd()))  # the notebooks live in code/analysis
import analysis_utils as au
importlib.reload(au)

RESULTS = pathlib.Path('results')
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

EXPS = au.available_experiments()
print('experiments with exported artifacts:', EXPS)
missing = [e for e in au.EXPERIMENTS if e not in EXPS]
if missing:
    print('NOT YET RERUN (no artifacts):', missing)

In [ ]:
MODEL = 'random_forest'
SPLIT = 'grouped'
LABEL = 'multi'

## 1. Packet outcome per device type and group

In [ ]:
rows = []
for e in EXPS:
    for frame in ['indomain', 'transfer']:
        p = au.load_predictions(e, LABEL, MODEL, SPLIT, frame)
        tgt = p[p['y_true'] != 0]
        for (dev, grp), g in tgt.groupby(['labelled_device', 'y_true']):
            rows.append(dict(exp=au.exp_name(e), frame=frame, device=dev, group=int(grp),
                             packets=len(g), sources=g['source'].nunique(),
                             own_group=(g['y_pred'] == grp).mean(),
                             other_group=((g['y_pred'] != grp) & (g['y_pred'] != 0)).mean(),
                             background=(g['y_pred'] == 0).mean()))
dev = pd.DataFrame(rows)
dev.to_csv(RESULTS / 'error_by_device.csv', index=False)
dev[dev['frame'] == 'transfer'].round(3)

In [ ]:
# Recall of the own group per device type on the university capture
h = (dev[dev['frame'] == 'transfer'].assign(unit=lambda d: d['device'] + ' G' + d['group'].astype(str))
     .pivot(index='unit', columns='exp', values='own_group'))
fig, ax = plt.subplots(figsize=(9, 0.45 * len(h) + 1.5))
im = ax.imshow(h.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(h.shape[1])); ax.set_xticklabels(h.columns, rotation=20, ha='right')
ax.set_yticks(range(h.shape[0])); ax.set_yticklabels(h.index)
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        if not np.isnan(h.values[i, j]):
            ax.text(j, i, f'{h.values[i, j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if h.values[i, j] < 0.5 else 'black')
fig.colorbar(im, label='share of packets assigned to the own group')
ax.set_title('University capture, random forest, grouped split')
plt.tight_layout(); plt.savefig(FIGURES / 'fig_error_by_device_transfer.pdf'); plt.show()

## 1b. The same per sublabel (e.g. phone scan requests against adverts)

In [ ]:
rows = []
for e in EXPS:
    p = au.load_predictions(e, LABEL, MODEL, SPLIT, 'transfer')
    tgt = p[p['y_true'] != 0].copy()
    tgt['sublabel_device'] = tgt['sublabel_device'].fillna('(none)')
    for (dev_, sub, grp), g in tgt.groupby(['labelled_device', 'sublabel_device', 'y_true']):
        rows.append(dict(exp=au.exp_name(e), device=dev_, sublabel=sub, group=int(grp), packets=len(g),
                         own_group=(g['y_pred'] == grp).mean(),
                         other_group=((g['y_pred'] != grp) & (g['y_pred'] != 0)).mean(),
                         background=(g['y_pred'] == 0).mean()))
pd.DataFrame(rows).round(3)

## 2. Per address (majority vote)

In [ ]:
rows = []
for e in EXPS:
    for frame in ['indomain', 'transfer']:
        p = au.load_predictions(e, LABEL, MODEL, SPLIT, frame)
        src = au.majority_vote(p, p['y_true'], p['y_pred'])
        for (d_, grp), g in src[src['y_true'] != 0].groupby(['labelled_device', 'y_true']):
            rows.append(dict(exp=au.exp_name(e), frame=frame, device=d_, group=int(grp), addresses=len(g),
                             correct=(g['y_pred'] == grp).sum(),
                             other_group=((g['y_pred'] != grp) & (g['y_pred'] != 0)).sum(),
                             background=(g['y_pred'] == 0).sum(),
                             median_packets_per_address=g['n_packets'].median()))
src_dev = pd.DataFrame(rows)
src_dev.to_csv(RESULTS / 'error_by_device_sources.csv', index=False)
src_dev[src_dev['frame'] == 'transfer']

## 3. False positives: background addresses predicted as a group

In [ ]:
rows = []
for e in EXPS:
    p = au.load_predictions(e, LABEL, MODEL, SPLIT, 'transfer')
    ev = au.load_frame(e, 'eval', columns=['company_id'])
    p['company_id'] = ev['company_id'].to_numpy()[p['row']]
    bg = p[p['y_true'] == 0]
    fp = bg[bg['y_pred'] != 0]
    by = (fp.groupby('company_id', dropna=False)
          .agg(fp_packets=('y_pred', 'size'), fp_sources=('source', 'nunique'),
               as_g1=('y_pred', lambda s: (s == 1).sum()), as_g2=('y_pred', lambda s: (s == 2).sum()))
          .sort_values('fp_packets', ascending=False).head(10).reset_index())
    by.insert(0, 'exp', au.exp_name(e))
    by['share_of_all_fp'] = by['fp_packets'] / max(len(fp), 1)
    by['bg_fp_rate'] = len(fp) / max(len(bg), 1)
    rows.append(by)
fps = pd.concat(rows, ignore_index=True)
fps.to_csv(RESULTS / 'false_positives_by_company.csv', index=False)
fps

## 4. Errors over time

In [ ]:
rows = []
for e in EXPS:
    p = au.load_predictions(e, LABEL, MODEL, SPLIT, 'indomain')
    for b, g in p.groupby('block'):
        t = g[g['y_true'] != 0]
        rows.append(dict(exp=au.exp_name(e), frame='indomain', window=f'block {int(b)}',
                         target_packets=len(t), own_group=(t['y_pred'] == t['y_true']).mean() if len(t) else np.nan))
    p = au.load_predictions(e, LABEL, MODEL, SPLIT, 'transfer')
    p['window'] = ((p['time'] - p['time'].min()) // 600).astype(int)
    for w, g in p.groupby('window'):
        t = g[g['y_true'] != 0]
        rows.append(dict(exp=au.exp_name(e), frame='transfer', window=f'{10 * w:03d}-{10 * w + 10:03d} min',
                         target_packets=len(t), own_group=(t['y_pred'] == t['y_true']).mean() if len(t) else np.nan))
over_time = pd.DataFrame(rows)
over_time.to_csv(RESULTS / 'error_over_time.csv', index=False)
over_time.pivot_table(index=['frame', 'window'], columns='exp', values='own_group').round(2)